# 1. Data Collection — SpaceX REST API

This notebook collects Falcon 9 launch history from the public SpaceX REST API,
flattens the nested JSON response, and does an initial filter down to Falcon 9
launches only (Falcon 1 excluded).

In [ ]:
import requests
import pandas as pd

# GET request against the SpaceX v4 launches endpoint
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print(response.status_code)

In [ ]:
# Flatten the nested JSON response into a DataFrame
data = pd.json_normalize(response.json())
data.head()

In [ ]:
# First row's static fire date (used to sanity-check the earliest launch record)
print(data['static_fire_date_utc'].iloc[0])
print("Year of first record:", pd.to_datetime(data['static_fire_date_utc'].iloc[0]).year)

2006-03-17T00:00:00.000Z
Year of first record: 2006


In [ ]:
# Keep only columns needed downstream, and expand list-valued columns
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# Remove rows with multiple cores/payloads (these are not standard Falcon 9 flights)
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Restrict to launches on/before a fixed cutoff so results are reproducible
data = data[data['date'] <= pd.to_datetime('2020-11-13').date()]

In [ ]:
# Look up rocket name via a second API call per unique rocket id, then filter to Falcon 9
def get_rocket_name(rocket_id):
    r = requests.get(f"https://api.spacexdata.com/v4/rockets/{rocket_id}").json()
    return r['name']

data['rocket_name'] = data['rocket'].apply(get_rocket_name)
falcon9_only = data[data['rocket_name'] == 'Falcon 9'].reset_index(drop=True)

print("Falcon 9 launches after removing Falcon 1:", len(falcon9_only))

Falcon 9 launches after removing Falcon 1: 90


Continued in **02_data_wrangling.ipynb**, where launchpad/payload/core ids
are resolved into readable fields (LaunchSite, PayloadMass, Orbit, LandingPad, etc.)
and the outcome label is derived.